<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
To install RapidFire AI on your own machine, see the <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide in our docs.
</div>

### RapidFire AI Tutorial Use Case: SFT for Scheme Linking

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

### Load Dataset 

In [8]:
import json
from pathlib import Path
from datasets import Dataset, load_dataset

SCHEMAS_DIR = Path("./schemas")
TRAIN_JSON_PATH = Path("./train.json")

def db_id_to_schema_path(db_id, schemas_dir=SCHEMAS_DIR):
    # change white space and forward slash to underscore
    filename = db_id.replace(" ", "_").replace("/", "_") + ".json"
    return schemas_dir / filename

def load_schema_as_dict(db_id, schemas_dir=SCHEMAS_DIR):
    schema_path = db_id_to_schema_path(db_id, schemas_dir)
    with open(schema_path, "r", encoding="utf-8") as f:
        schema_json = json.load(f)

    table_names = schema_json["table_names_original"]
    schema = {table_name: [] for table_name in table_names}

    for table_idx, column_name in schema_json["column_names_original"]:
        if table_idx == -1:  # skip synthetic "*"
            continue
        schema[table_names[table_idx]].append(column_name)

    return schema

def compact_schema_text(schema):
    # table(column, column, ...)
    lines = []
    for table_name, columns in schema.items():
        columns_text = ", ".join(columns)
        lines.append(f"{table_name}({columns_text})")
    return "\n".join(lines)

with open(TRAIN_JSON_PATH, "r", encoding="utf-8") as f:
    raw_train_data = json.load(f)

processed_train_data = []

for example in raw_train_data:
    schema_path = db_id_to_schema_path(example["db_id"])

    schema_dict = load_schema_as_dict(example["db_id"])
    processed_train_data.append({
        "question_id": example["question_id"],
        "db_id": example["db_id"],
        "question": example["question"],
        "schema_text": compact_schema_text(schema_dict),
        "schema_links_json": json.dumps(
            example["schema_links"],
            ensure_ascii=False,
            sort_keys=True,
        ),
    })

train_dataset = Dataset.from_list(processed_train_data).shuffle(seed=42)

print(f"Loaded {len(train_dataset)} training examples.")

Loaded 301 training examples.


In [9]:
VALIDATION_JSON_PATH = Path("./validation.json")  

with open(VALIDATION_JSON_PATH, "r", encoding="utf-8") as f:
    raw_eval_data = json.load(f)

processed_eval_data = []

for example in raw_eval_data:
    schema_path = db_id_to_schema_path(example["db_id"])

    schema_dict = load_schema_as_dict(example["db_id"])
    processed_eval_data.append({
        "question_id": example["question_id"],
        "db_id": example["db_id"],
        "question": example["question"],
        "schema_text": compact_schema_text(schema_dict),
        "schema_links_json": json.dumps(
            example["schema_links"],
            ensure_ascii=False,
            sort_keys=True,
        ),
    })

eval_dataset = Dataset.from_list(processed_eval_data).shuffle(seed=42)

print(f"Loaded {len(eval_dataset)} eval examples.")

Loaded 101 eval examples.


In [12]:
import json

def print_dataset_examples(dataset, dataset_name, num_examples=3, show_schema_max_chars=800):
    print("=" * 100)
    print(f"{dataset_name}: {len(dataset)} examples")
    print("=" * 100)

    for i in range(min(num_examples, len(dataset))):
        example = dataset[i]

        print(f"\n[{dataset_name} example {i}]")
        print("-" * 100)

        print("question_id:", example.get("question_id"))
        print("db_id:", example.get("db_id"))

        print("\nQuestion:")
        print(example.get("question"))

        print("\nSchema:")
        schema_text = example.get("schema_text", "")
        print(schema_text[:show_schema_max_chars])
        if len(schema_text) > show_schema_max_chars:
            print(f"... [truncated, total {len(schema_text)} chars]")

        print("\nSchema links:")
        print(json.dumps(
            json.loads(example["schema_links_json"]),
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        ))


print_dataset_examples(train_dataset, "TRAIN", num_examples=1)
print_dataset_examples(eval_dataset, "EVAL", num_examples=1)

TRAIN: 301 examples

[TRAIN example 0]
----------------------------------------------------------------------------------------------------
question_id: 210
db_id: SBODemoUS-Service

Question:
What is the renewal status of the contracts with the customer named 'Maxi-Teq'? Include the contract ID and the status in the result.

Schema:
OCTT(TmpltName, Deleted, Renewal, RemindVal, RemindUnit, Duration, ResponsVal, ResponsUnt, Descriptio, PriceList, CntrctType, MonEnabled, TueEnabled, WedEnabled, ThuEnabled, FriEnabled, SatEnabled, SunEnabled, MonStart, MonEnd, TueStart, TueEnd, WedStart, WedEnd, ThuStart, ThuEnd, FriStart, FriEnd, SatStart, SatEnd, SunStrart, SunEnd, InclParts, InclWork, InclTravel, Attachment, CreateDate, Remark, InclHldays, ResponseV, ResponseU, AtcEntry)
OSCS(statusID, Name, Descriptio, Locked, Active)
OINS(insID, customer, custmrName, contactCod, directCsmr, drctCsmNam, manufSN, internalSN, warranty, wrrntyStrt, wrrntyEnd, responsVal, responsUnt, itemCode, itemName, i

### Define Data Processing Function

In [ ]:
def sample_formatting_function(row):
    """Format schema-linking train/eval examples."""

    system_prompt = (
        "You are a schema linking assistant. "
        "Given a database schema and a natural language question, "
        "identify the database tables and columns needed to answer the question. "
        "Return only valid JSON. Do not include explanations."
    )

    user_prompt = f"""Database:
{row["db_id"]}

Schema:
{row["schema_text"]}

Question:
{row["question"]}

Output format:
{{"TABLE_NAME": ["COLUMN_NAME_1", "COLUMN_NAME_2"], "TABLE_ONLY": []}}

Rules:
- Use only table and column names that appear in the schema.
- Include a table with an empty list if the table is referenced but no specific column is needed.
- Do not include SQL.
- Return only the JSON object."""

    completion_text = row["schema_links_json"]

    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "completion": [
            {"role": "assistant", "content": completion_text}
        ],
    }

### Initialize Experiment

In [ ]:
# Every experiment instance must be uniquely named
experiment = Experiment(experiment_name="exp1-chatqa-lite", mode="fit")

### Define Custom Eval Metrics Function

In [ ]:
def sample_compute_metrics(eval_preds):  
    """Optional function to compute eval metrics based on predictions and labels"""
    predictions, labels = eval_preds

    # Standard text-based eval metrics: Rouge and BLEU
    import evaluate
    rouge = evaluate.load("rouge")
    bleu = evaluate.load("bleu")

    rouge_output = rouge.compute(predictions=predictions, references=labels, use_stemmer=True)
    rouge_l = rouge_output["rougeL"]
    bleu_output = bleu.compute(predictions=predictions, references=labels)
    bleu_score = bleu_output["bleu"]

    return {
        "rougeL": round(rouge_l, 4),
        "bleu": round(bleu_score, 4),
    }

### Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [ ]:
# 2 LoRA PEFT configs lite with different adapter capacities
peft_configs_lite = List([
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],  # Standard transformer naming
        bias="none"
    ),
    RFLoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Standard naming
        bias="none"
    )
])

# 2 base models x 2 peft configs = 4 combinations in total
config_set_lite = List([
    RFModelConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # 1.1B model
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-3,  # Higher LR for very small model
            lr_scheduler_type="linear",
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            max_steps=128,
            gradient_accumulation_steps=1,   # No accumulation needed
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            bf16=True,
        ),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 256,
            "temperature": 0.8,  # Higher temp for tiny model
            "top_p": 0.9,
            "top_k": 30,         # Reduced top_k
            "repetition_penalty": 1.05,
        }
    ),
    RFModelConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # 1.1B model
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-4,  # Higher LR for very small model
            lr_scheduler_type="linear",
            per_device_train_batch_size=4,  # Even larger batch size
            per_device_eval_batch_size=4,
            max_steps=128,
            gradient_accumulation_steps=1,   # No accumulation needed
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            bf16=True,
        ),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 256,
            "temperature": 0.8,  # Higher temp for tiny model
            "top_p": 0.9,
            "top_k": 30,         # Reduced top_k
            "repetition_penalty": 1.05,
        }
    )
])


#### Define Model Creation Function for All Model Types Across Configs

In [ ]:

def sample_create_model(model_config): 
     """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
     from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM

     model_name = model_config["model_name"]
     model_type = model_config["model_type"]
     model_kwargs = model_config["model_kwargs"]
 
     if model_type == "causal_lm":
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "seq2seq_lm":
          model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "masked_lm":
          model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "custom":
          # Handle custom model loading logic, e.g., loading your own checkpoints
          # model = ... 
          pass
     else:
          # Default to causal LM
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
      
     tokenizer = AutoTokenizer.from_pretrained(model_name)
      
     return (model,tokenizer)


#### Generate Config Group

In [ ]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type="SFT"
)

### Run Multi-Config Training

In [ ]:
# Launch training of all configs in the config_group with swap granularity of 4 chunks
experiment.run_fit(config_group, sample_create_model, train_dataset, eval_dataset, num_chunks=4, seed=42)

### End Current Experiment

In [ ]:
experiment.end()

<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Thanks for trying RapidFire AI! ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>